# 🕹️ Visium HD Game Tutorial — Spatial Sprite Hunt

Welcome to the **spatial transcriptomics arcade**! This notebook is a playful companion to the main [CRC Visium HD tutorial](./visium_hd_crc_p1_tutorial.ipynb). Instead of a real patient sample you get a synthetic tissue with **7 retro arcade sprites hidden inside**. Each time you correctly perform a core analysis step, another sprite gets revealed and you earn points.

**How to play**
- Run each cell top to bottom.
- Some cells are automatic — they reveal a sprite for free once the prior step is done.
- Cells marked **🎯 TODO** need you to write a small piece of code. A validator checks your answer. If correct, a sprite appears and your score ticks up. If not, you get a hint and can try again.

**Game map (mirrors the main tutorial's sections)**

| Step | Main tutorial section | Reveal |
|---|---|---|
| 1 | §1 Data Loading | ⭐ Star |
| 2 | §3 QC — per-bin metrics 🎯 | ❤️ Heart |
| 3 | §5 Filtering & Normalization | 🍄 Mushroom |
| 4 | §5 Clustering | 👾 Pac-Man |
| 5 | §7 Cell-type Annotation 🎯 | 👻 Ghost |
| 6 | §8 Neighborhoods | 🛸 Space Invader |
| 7 | §9 Spatially Variable Genes 🎯 | 👽 Alien |

Ready? Insert coin to continue ➡️


## 0. Setup

Installs the few dependencies the game needs and makes the `scripts/` directory importable. On Colab this pulls the repo so you get `game_utils.py` for free.

In [ ]:
# --- Dependency install (Colab-friendly) ---
import sys, subprocess, os

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

try:
    import scanpy, anndata  # noqa: F401
except ImportError:
    _pip("scanpy", "anndata", "leidenalg", "python-igraph")

# Clone the repo so game_utils.py is importable on a fresh Colab runtime.
if not os.path.exists("spatial_transcriptomics_resources"):
    if "google.colab" in sys.modules:
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/stevenpastor/spatial_transcriptomics_resources.git"],
            check=True,
        )
        sys.path.insert(0, "spatial_transcriptomics_resources/scripts")
    else:
        # Running locally from notebook/ — step up one dir.
        sys.path.insert(0, os.path.abspath("../scripts"))
else:
    sys.path.insert(0, "spatial_transcriptomics_resources/scripts")

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, scanpy as sc, matplotlib.pyplot as plt
sc.settings.verbosity = 1
print("ready to play 🎮")


## Step 1 — Load the synthetic tissue  ⭐

> **Main tutorial parallel:** §1 Data Loading

A real Visium HD dataset gives you a bin × gene count matrix on a physical tissue grid. Our synthetic tissue works the same way: a `100 × 100` grid of bins, ~100 genes, and a few hundred "sprite" bins hiding inside a sea of noise.

Run the next cell to build it and earn your first sprite — a **⭐ Star** — just for loading.

In [ ]:
from game_utils import (
    make_synthetic_tissue, GameState, show_raw_tissue,
    validate_load, validate_qc, validate_normalization,
    validate_clusters, validate_annotation, validate_neighborhoods,
    validate_svgs, build_spatial_graph, morans_i, SPRITE_ORDER,
)

game = GameState()
adata_raw = make_synthetic_tissue(seed=42)
validate_load(adata_raw, game)
adata_raw


### What does the raw tissue look like?

In a Visium HD experiment the first thing you do is plot total counts per bin to eyeball tissue quality. Let's do that now. Spoiler: the sprites are in there, but background noise is hiding them.

In [ ]:
show_raw_tissue(adata_raw)

## Step 2 — Quality control 🎯 ❤️

> **Main tutorial parallel:** §3 QC — Per-Bin Metrics

Compute per-bin QC metrics (`total_counts`, `n_genes`, `pct_mt`) and filter out low-quality bins. Background bins have **low counts** and **high mitochondrial percentage** — exactly the pattern you see in a real FFPE Visium HD sample.

**🎯 TODO:** pick thresholds that strip the background while keeping the sprite bins. The validator wants ≥80% of the background removed *and* ≥80% of sprite bins retained.

In [ ]:
# Compute QC metrics on the raw tissue
X = np.asarray(adata_raw.X)
adata_raw.obs["total_counts"] = X.sum(axis=1)
adata_raw.obs["n_genes"] = (X > 0).sum(axis=1)
mt_mask = adata_raw.var["mt"].values
adata_raw.obs["pct_mt"] = X[:, mt_mask].sum(axis=1) / adata_raw.obs["total_counts"].clip(lower=1)

# Quick look at the distributions
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, col in zip(axes, ["total_counts", "n_genes", "pct_mt"]):
    ax.hist(adata_raw.obs[col], bins=60, color="#30D158", log=True)
    ax.set_title(col)
plt.tight_layout(); plt.show()


In [ ]:
### 🎯 YOUR CODE HERE — set QC thresholds ###
# Hint: background bins have total_counts well below ~60 and pct_mt above ~0.2.
MIN_COUNTS = ...    # e.g. 100
MAX_PCT_MT = ...    # e.g. 0.1

keep = (adata_raw.obs["total_counts"] >= MIN_COUNTS) & (adata_raw.obs["pct_mt"] < MAX_PCT_MT)
adata = adata_raw[keep].copy()
print(f"kept {adata.n_obs}/{adata_raw.n_obs} bins")

validate_qc(adata, adata_raw, game)


## Step 3 — Normalize  🍄

> **Main tutorial parallel:** §5 Filtering & Normalization

Raw counts are hard to compare across bins. A standard scanpy pipeline normalizes each bin to a fixed library size then log-transforms. This step is automatic — run it to unlock the **🍄 Mushroom**.

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
validate_normalization(adata, game)


## Step 4 — Cluster the tissue  👾

> **Main tutorial parallel:** §5 Clustering (Leiden after PCA + neighbor graph)

PCA → kNN graph → Leiden clustering. Each sprite has its own marker genes, so a good clustering should assign one cluster per sprite. The validator wants ≥5 of the 7 sprite regions matched by a cluster.

In [ ]:
sc.pp.pca(adata, n_comps=20)
sc.pp.neighbors(adata, n_neighbors=10)
sc.tl.leiden(adata, resolution=1.2)

print("clusters:", adata.obs["leiden"].nunique())
validate_clusters(adata, "leiden", game)


## Step 5 — Marker-based cell type annotation 🎯 👻

> **Main tutorial parallel:** §7 Marker-Based Annotation

In the main tutorial you assign cell types by picking marker genes. Here each sprite is its own "cell type" with 3 dedicated marker genes. Your job: correctly identify the **Ghost** (`GhostCell`) markers.

**🎯 TODO:** fill in `marker_dict["GhostCell"]` with the three gene names that light up the ghost region. Tip — the gene name prefix is the sprite's name. You can also eyeball the cluster that corresponds to the ghost and list its top differentially-expressed genes.

In [ ]:
# Helper: show top genes per cluster so you can identify which cluster is the ghost
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon", n_genes=5)
top = pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(5)
print(top)


In [ ]:
### 🎯 YOUR CODE HERE — name the ghost markers ###
marker_dict = {
    "GhostCell": [..., ..., ...],  # fill in 3 gene names
}

validate_annotation(adata, marker_dict, game)


## Step 6 — Spatial neighborhoods 🛸

> **Main tutorial parallel:** §8 Neighborhood Analysis

Build a spatial neighbor graph from bin coordinates. This underpins everything spatial in squidpy (co-occurrence, enrichment, SVGs). The helper below uses a radius-based cKDTree — same idea as `sq.gr.spatial_neighbors`, just minimal.

In [ ]:
build_spatial_graph(adata, radius=1.5)
validate_neighborhoods(adata, game)


## Step 7 — Spatially variable genes 🎯 👽

> **Main tutorial parallel:** §9 Spatially Variable Genes

The last sprite is the stealthiest: the **👽 Alien** lives in a tiny region and only its three marker genes have a distinct spatial pattern there. Compute Moran's I across all genes and pick enough top hits to capture the alien's markers.

**🎯 TODO:** choose `TOP_N` large enough that at least 2 of the 3 alien markers are in your SVG list.

In [ ]:
### 🎯 YOUR CODE HERE — pick how many SVGs to keep ###
TOP_N = ...   # small numbers miss the alien; ~20 is plenty

svg_results = morans_i(adata, top_n=TOP_N)
svg_list = [g for g, _ in svg_results]
print("top SVGs:", svg_list)

validate_svgs(svg_list, adata, game)


## 🏆 Scoreboard

In [ ]:
print(game.banner())
print(game.final_message())
print()
print("Sprites unlocked:")
for s in SPRITE_ORDER:
    check = "✅" if s in game.unlocked else "⬜"
    print(f"  {check}  {s}")


### What just happened?

Every step you ran has an exact counterpart in the [main CRC tutorial](./visium_hd_crc_p1_tutorial.ipynb):

- **QC thresholds** → real tissue has the same bimodal distribution. The cutoffs you found here are the *same* kind of cutoffs you tune for a real sample.
- **Leiden clustering** → identifies tissue zones. On real data those zones correspond to tumor vs stroma vs immune niches.
- **Marker-based annotation** → in the real notebook you compare marker scores against a published ground truth. Here the ground truth is the sprite layout.
- **Spatial neighbor graph** → backbone of every spatial statistic you'll ever compute.
- **Spatially variable genes** → finds genes whose expression *clusters in space*. On CRC data those are things like `KRT19` outlining tumor glands.

Head back to the main tutorial and do it on real tissue. 🧪
